# 🏥 ATM-Net++: Lumbar Spine MRI Segmentation
### Target: Val Dice > 0.85 on Tesla T4

## ✅ Before running:
1. **Runtime → Change runtime type → T4 GPU**
2. Your dataset files (`images/`, `masks/`, `overview.csv`) must already be uploaded
3. **Run All Cells** (Runtime → Run all)

---
Everything is automated. Just run all cells and wait.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: Check GPU
# ═══════════════════════════════════════════════════════════════
import torch
assert torch.cuda.is_available(), 'NO GPU! Go to Runtime > Change runtime type > T4 GPU'
print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM: {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB')
print(f'✓ PyTorch: {torch.__version__}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: Install dependencies
# ═══════════════════════════════════════════════════════════════
!pip install SimpleITK opencv-python-headless scipy pandas -q
print('✓ Dependencies ready')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: Mount Google Drive (to save checkpoints)
# ═══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/ATM_Net_outputs', exist_ok=True)
print('✓ Drive mounted. Checkpoints will be saved to: /content/drive/MyDrive/ATM_Net_outputs/')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: Fix dataset paths (auto-detects where files are)
# ═══════════════════════════════════════════════════════════════
import shutil, glob, os
from pathlib import Path

os.makedirs('/content/SPIDER/images', exist_ok=True)
os.makedirs('/content/SPIDER/masks',  exist_ok=True)

# Search everywhere for .mha files and move to correct location
search_paths = [
    '/content/images/**/*.mha', '/content/images/*.mha',
    '/content/**/*.mha',  # broad search
]

img_moved = 0
for pattern in search_paths:
    for f in glob.glob(pattern, recursive=True):
        name = Path(f).name
        if '_t' not in name: continue  # skip non-patient files
        # Decide: image or mask? Masks dir takes priority if file exists in both
        dst_img  = f'/content/SPIDER/images/{name}'
        if not Path(dst_img).exists():
            shutil.copy2(f, dst_img)
            img_moved += 1

msk_moved = 0
for pattern in ['/content/masks/**/*.mha', '/content/masks/*.mha']:
    for f in glob.glob(pattern, recursive=True):
        dst = f'/content/SPIDER/masks/{Path(f).name}'
        if not Path(dst).exists():
            shutil.copy2(f, dst)
            msk_moved += 1

# Copy CSV files
for csv in ['overview.csv', 'radiological_gradings.csv']:
    for src in [f'/content/{csv}', f'/content/images/{csv}',
                '/content/drive/MyDrive/SPIDER/10159290/'+csv]:
        if Path(src).exists():
            shutil.copy2(src, f'/content/SPIDER/{csv}')
            print(f'✓ Copied {csv}')
            break

n_img = len(glob.glob('/content/SPIDER/images/*.mha'))
n_msk = len(glob.glob('/content/SPIDER/masks/*.mha'))
has_csv = Path('/content/SPIDER/overview.csv').exists()

print(f'\n✓ Images : {n_img} files')
print(f'✓ Masks  : {n_msk} files')
print(f'✓ CSV    : {has_csv}')

if n_img < 10:
    print('\n⚠ ERROR: Images not found!')
    print('Please upload images.zip and run: !unzip images.zip -d /content/SPIDER/images/')
    print('\nSearching for any .mha files...')
    found = glob.glob('/content/**/*.mha', recursive=True)[:10]
    for f in found: print(f'  Found: {f}')
else:
    print(f'\n✓ Dataset ready!')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5: FULL TRAINING (runs all 200 epochs automatically)
# Expected: ~45 sec/epoch on T4 = ~2.5 hours total
# Checkpoints saved to Google Drive every epoch if new best
# ═══════════════════════════════════════════════════════════════
import sys, os, time, warnings, json, random, gc
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, cv2
import SimpleITK as sitk
from pathlib import Path
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Config
DATA_ROOT   = Path('/content/SPIDER')
IMAGES_DIR  = DATA_ROOT / 'images'
MASKS_DIR   = DATA_ROOT / 'masks'
OVERVIEW    = DATA_ROOT / 'overview.csv'
OUTPUT_DIR  = Path('/content/drive/MyDrive/ATM_Net_outputs')
CKPT_BEST   = OUTPUT_DIR / 'best_model.pth'
CACHE_DIR   = Path('/content/cache'); CACHE_DIR.mkdir(exist_ok=True)

IMG_SIZE=384; BATCH_SIZE=8; EPOCHS=200; LR=2e-4; WD=5e-4; MAX_SPP=30; NC=19; SEED=42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

S2A = {**{i:i for i in range(1,9)}, 100:9, **{201+i:10+i for i in range(8)}}
CN  = {0:'bg',1:'Vert-1',2:'Vert-2',3:'Vert-3',4:'Vert-4',5:'Vert-5',6:'Vert-6',
       7:'Vert-7',8:'Vert-8',9:'Sacrum',10:'IVD-1',11:'IVD-2',12:'IVD-3',
       13:'IVD-4',14:'IVD-5',15:'IVD-6',16:'IVD-7',17:'IVD-8',18:'Canal'}
RARE=[7,8,16,17]
CW=torch.tensor([0,1,1,1,1.5,2,3,6,12,1,6,4,4,5,6,9,14,30,0]).float()

def remap(m):
    o=np.zeros_like(m,dtype=np.int32)
    for s,d in S2A.items(): o[m==s]=d
    return o

def fg(m): return float((m>0).sum())/max(m.size,1)

def build_cache(pids,split):
    cf=CACHE_DIR/f'{split}_{IMG_SIZE}.npz'
    if cf.exists():
        d=np.load(cf); n=len(d['imgs'])
        print(f'  {split}: {n} slices (cached)'); return d['imgs'],d['msks'],d['rare'].tolist()
    print(f'  {split}: processing {len(pids)} patients...')
    imgs,msks,rare=[],[],[]; skipped=0
    for i,pid in enumerate(pids):
        ip=IMAGES_DIR/f'{pid}_t2.mha'; mp=MASKS_DIR/f'{pid}_t2.mha'
        if not ip.exists() or not mp.exists(): skipped+=1; continue
        try:
            iv=sitk.GetArrayFromImage(sitk.ReadImage(str(ip))).astype(np.float32)
            mv=sitk.GetArrayFromImage(sitk.ReadImage(str(mp))).astype(np.int32)
        except: skipped+=1; continue
        n=iv.shape[0]; lo,hi=int(n*0.05),int(n*0.95)
        ranked=sorted(range(lo,hi),key=lambda s:fg(remap(mv[s])),reverse=True)[:MAX_SPP]
        for s in ranked:
            rm=remap(mv[s])
            if fg(rm)<0.005: continue
            p1,p99=np.percentile(iv[s],[0.5,99.5])
            ir=cv2.resize(np.clip((iv[s]-p1)/(p99-p1+1e-8),0,1).astype(np.float32),
                          (IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR).astype(np.float16)
            mr=cv2.resize(rm.astype(np.float32),(IMG_SIZE,IMG_SIZE),
                          interpolation=cv2.INTER_NEAREST).astype(np.uint8)
            imgs.append(ir); msks.append(np.clip(mr,0,NC-1))
            rare.append(1.0 if float(sum((rm==c).sum() for c in RARE))/max(rm.size,1)>0.0003 else 0.1)
        if (i+1)%30==0: print(f'    {i+1}/{len(pids)}, {len(imgs)} slices...')
    if not imgs: raise ValueError(f'No slices loaded! Found 0 usable files in {IMAGES_DIR}')
    np.savez_compressed(cf,imgs=np.stack(imgs).astype(np.float16),
                        msks=np.stack(msks).astype(np.uint8),rare=np.array(rare,dtype=np.float32))
    print(f'  {split}: {len(imgs)} slices ready ({skipped} skipped)')
    return np.stack(imgs).astype(np.float16),np.stack(msks).astype(np.uint8),rare

class Aug:
    def __call__(self,img,msk):
        if random.random()<0.5: img=np.fliplr(img).copy();msk=np.fliplr(msk).copy()
        if random.random()<0.6:
            M=cv2.getRotationMatrix2D((IMG_SIZE//2,IMG_SIZE//2),random.uniform(-20,20),1)
            img=cv2.warpAffine(img.astype(np.float32),M,(IMG_SIZE,IMG_SIZE),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT)
            mf=cv2.warpAffine(msk.astype(np.float32),M,(IMG_SIZE,IMG_SIZE),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            msk=np.clip(mf.astype(np.int32),0,NC-1)
        img=np.clip(np.power(img.astype(np.float32)+1e-8,random.uniform(0.65,1.5)),0,1)
        img=np.clip(img*random.uniform(0.75,1.25)+random.uniform(-0.1,0.1),0,1)
        return img.astype(np.float32),msk.astype(np.int64)

class DS(Dataset):
    def __init__(self,i,m,a=None): self.i=i;self.m=m;self.a=a
    def __len__(self): return len(self.i)
    def __getitem__(self,x):
        img=self.i[x].astype(np.float32);msk=self.m[x].astype(np.int64)
        if self.a: img,msk=self.a(img,msk)
        return torch.from_numpy(img[None]).float(),torch.from_numpy(msk).long()

class CA(nn.Module):
    def __init__(self,ch,r=8):
        super().__init__();r=max(1,ch//r)
        self.a=nn.AdaptiveAvgPool2d(1);self.b=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Flatten(),nn.Linear(ch,r),nn.ReLU(True),nn.Linear(r,ch),nn.Sigmoid())
    def forward(self,x): return x*(self.fc(self.a(x))+self.fc(self.b(x))).clamp(0,1).view(x.shape[0],-1,1,1)

class SA(nn.Module):
    def __init__(self):
        super().__init__()
        self.c=nn.Sequential(nn.Conv2d(2,1,7,padding=3,bias=False),nn.BatchNorm2d(1),nn.Sigmoid())
    def forward(self,x): return x*self.c(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1))

class RB(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.n=nn.Sequential(nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch),nn.ReLU(True),
                             nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch))
        self.ca=CA(ch);self.sa=SA();self.act=nn.ReLU(True)
    def forward(self,x): return self.act(self.sa(self.ca(self.n(x)))+x)

class Enc(nn.Module):
    def __init__(self,ci,co,d=0):
        super().__init__()
        self.c=nn.Sequential(nn.Conv2d(ci,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True),
                             nn.Conv2d(co,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True))
        self.r=RB(co);self.d=nn.Dropout2d(d) if d>0 else nn.Identity()
    def forward(self,x): return self.d(self.r(self.c(x)))

class Net(nn.Module):
    def __init__(self,b=32,nc=19,d=0.2):
        super().__init__()
        self.e1=Enc(1,b);self.e2=Enc(b,b*2,d*0.3);self.e3=Enc(b*2,b*4,d*0.6);self.e4=Enc(b*4,b*8,d*0.8)
        self.bn=nn.Sequential(Enc(b*8,b*16,d),nn.Dropout2d(d));self.p=nn.MaxPool2d(2)
        self.u4=nn.ConvTranspose2d(b*16,b*8,2,2);self.d4=Enc(b*16,b*8,d*0.4)
        self.u3=nn.ConvTranspose2d(b*8,b*4,2,2);self.d3=Enc(b*8,b*4,d*0.2)
        self.u2=nn.ConvTranspose2d(b*4,b*2,2,2);self.d2=Enc(b*4,b*2)
        self.u1=nn.ConvTranspose2d(b*2,b,2,2);self.d1=Enc(b*2,b)
        self.s3=nn.Conv2d(b*4,nc,1);self.s2=nn.Conv2d(b*2,nc,1);self.o=nn.Conv2d(b,nc,1)
        self.ax=nn.Sequential(nn.Conv2d(b,b,3,1,1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,nc,1))
    def forward(self,x):
        sz=x.shape[2:]
        e1=self.e1(x);e2=self.e2(self.p(e1));e3=self.e3(self.p(e2));e4=self.e4(self.p(e3))
        d=self.bn(self.p(e4))
        d=self.d4(torch.cat([self.u4(d),e4],1));d=self.d3(torch.cat([self.u3(d),e3],1))
        o3=F.interpolate(self.s3(d),sz,mode='bilinear',align_corners=False)
        d=self.d2(torch.cat([self.u2(d),e2],1))
        o2=F.interpolate(self.s2(d),sz,mode='bilinear',align_corners=False)
        d=self.d1(torch.cat([self.u1(d),e1],1))
        return (self.o(d),o2,o3,self.ax(d)) if self.training else self.o(d)

def dice_w(lg,tg,sm=1e-6):
    B,C,H,W=lg.shape;s=F.softmax(lg,1);o=F.one_hot(tg.clamp(0,C-1),C).permute(0,3,1,2).float()
    p=s[:,1:].reshape(B,C-1,-1);t=o[:,1:].reshape(B,C-1,-1)
    inter=(p*t).sum(-1);union=p.sum(-1)+t.sum(-1);mask=(t.sum(-1)>0).float()
    w=CW[1:].to(lg.device).view(1,C-1)
    return 1-((2*inter+sm)/(union+sm)*mask*w).sum()/(mask*w).sum().clamp(min=1)

def focal(lg,tg,g=2):
    ce=F.cross_entropy(lg,tg.clamp(0,NC-1),reduction='none'); return ((1-torch.exp(-ce))**g*ce).mean()

def boundary(lg,tg):
    s=F.softmax(lg,1);o=F.one_hot(tg.clamp(0,NC-1),NC).permute(0,3,1,2).float()
    b=(F.max_pool2d(o[:,1:],3,stride=1,padding=1)-o[:,1:]).clamp(0,1)
    w=CW[1:].to(lg.device).view(1,-1,1,1)
    return (b*(1-s[:,1:])*w).sum()/((b*w).sum()+1e-6)

def comp(lg,tg):
    tc=tg.clamp(0,NC-1)
    return F.cross_entropy(lg,tc,label_smoothing=0.03)+dice_w(lg,tc)+0.3*focal(lg,tc)+0.15*boundary(lg,tc)

def loss_fn(outs,tg):
    o1,o2,o3,ax=outs;main=comp(o1,tg)+0.3*comp(o2,tg)+0.15*comp(o3,tg)
    tc=tg.clamp(0,NC-1);rm=sum((tc==c).float() for c in RARE).clamp(0,1)
    return main+0.3*(F.cross_entropy(ax,tc,reduction='none')*(1+4*rm)).mean()

@torch.no_grad()
def get_dice(lg, tg):
    """Pure GPU dice — ~200x faster than CPU numpy loop."""
    B, C, H, W = lg.shape
    pred = lg.argmax(1)   # stays on GPU
    sm = 1e-6; D = defaultdict(list)
    for c in range(1, NC):
        p = (pred == c).float().view(B, -1)
        t = (tg   == c).float().view(B, -1)
        mask = (t.sum(1) > 0) | (p.sum(1) > 0)
        if not mask.any(): continue
        tp  = (p * t).sum(1)[mask]
        den = (p.sum(1) + t.sum(1))[mask]
        D[c].extend(((2*tp+sm)/(den+sm)).cpu().tolist())
    all_d = [v for vs in D.values() for v in vs]
    return D, float(np.mean(all_d)) if all_d else 0

def get_splits():
    df=pd.read_csv(OVERVIEW);tr,va=[],[]; seen=set()
    for n in df['new_file_name'].tolist():
        if not n.endswith('_t2') or 'SPACE' in n: continue
        p=n.replace('_t2','')
        if p in seen: continue
        seen.add(p);s=df.loc[df['new_file_name']==n,'subset'].values
        (va if len(s) and s[0]=='validation' else tr).append(p)
    return tr,va

# ── MAIN TRAINING LOOP ────────────────────────────────────
device=torch.device('cuda')
print('='*60)
print(f'  ATM-Net++ Training | GPU: {torch.cuda.get_device_name(0)}')
print(f'  {IMG_SIZE}x{IMG_SIZE} | BS={BATCH_SIZE} | LR={LR} | AMP=ON')
print('='*60)

tr_pids,va_pids=get_splits()
print(f'\nPatients: {len(tr_pids)} train | {len(va_pids)} val')
found=sum(1 for p in tr_pids if (IMAGES_DIR/f'{p}_t2.mha').exists())
print(f'Files found: {found}/{len(tr_pids)}')
if found==0:
    print('ERROR: No files found! Re-run Step 4 (path fix cell)')
    raise SystemExit

print('\nBuilding cache...')
ti,tm,tr_rare=build_cache(tr_pids,'train')
vi,vm,va_rare=build_cache(va_pids,'val')
print(f'RAM: ~{(ti.nbytes+tm.nbytes+vi.nbytes+vm.nbytes)//1024**2} MB\n')

sampler=WeightedRandomSampler(torch.tensor(tr_rare),len(tr_rare),replacement=True)
tr_dl=DataLoader(DS(ti,tm,Aug()),batch_size=BATCH_SIZE,sampler=sampler,num_workers=2,pin_memory=True)
va_dl=DataLoader(DS(vi,vm),batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)

model=Net(b=32,nc=NC,d=0.2).to(device)
np_=sum(p.numel() for p in model.parameters())
print(f'Model: {np_/1e6:.2f}M params | Batches: {len(tr_dl)} train | {len(va_dl)} val\n')

start_ep=1;best=0.0
if CKPT_BEST.exists():
    ck=torch.load(str(CKPT_BEST),map_location=device)
    model.load_state_dict(ck['model_state_dict'],strict=False)
    best=ck.get('best_dice',0);start_ep=ck.get('epoch',0)+1
    print(f'Resumed from epoch {ck.get("epoch")} | best={best:.4f}\n')

optim=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
sched=torch.optim.lr_scheduler.CosineAnnealingLR(optim,T_max=EPOCHS,eta_min=1e-6)
scaler=GradScaler();no_imp=0;t0tot=time.time()

print(f'{"Ep":>4}  {"TrDice":>8}  {"VaDice":>8}  {"Best":>8}  {"Gap":>6}  {"Sec":>5}')
print('─'*52)

for ep in range(start_ep,EPOCHS+1):
    model.train();td_b=[];t0=time.time();optim.zero_grad(set_to_none=True)
    for imgs,msks in tr_dl:
        imgs,msks=imgs.to(device,non_blocking=True),msks.to(device,non_blocking=True)
        with autocast(): outs=model(imgs);loss=loss_fn(outs,msks)
        scaler.scale(loss).backward()
        scaler.unscale_(optim);nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(optim);scaler.update();optim.zero_grad(set_to_none=True)
        with torch.no_grad(): _,d=get_dice(outs[0],msks);td_b.append(d)
    sched.step();td=float(np.mean(td_b));ep_s=time.time()-t0

    model.eval();Dc=defaultdict(list)
    with torch.no_grad():
        for imgs,msks in va_dl:
            imgs,msks=imgs.to(device,non_blocking=True),msks.to(device,non_blocking=True)
            with autocast(): out=model(imgs)
            D,_=get_dice(out,msks)
            for c,v in D.items(): Dc[c].extend(v)
    all_v=[v for vs in Dc.values() for v in vs]
    vd=float(np.mean(all_v)) if all_v else 0;gap=td-vd

    if vd>best:
        best=vd;no_imp=0
        pc={CN[c]:float(np.mean(v)) for c,v in Dc.items() if v}
        torch.save({'epoch':ep,'model_state_dict':model.state_dict(),'best_dice':best,'per_class_dice':pc},CKPT_BEST)
    else: no_imp+=1

    star='  ★' if vd==best else ''
    print(f'{ep:>4}  {td:>8.4f}  {vd:>8.4f}  {best:>8.4f}  {gap:>+6.3f}  {ep_s:>4.0f}s{star}')
    if vd>=0.90: print('\n🎯 TARGET ACHIEVED: Dice >= 0.90!'); break
    if vd>=0.85: print(f'  📈 Dice {vd:.4f} >= 0.85!')
    if no_imp>=50: print('\nEarly stop'); break

print(f'\n✓ Done: {(time.time()-t0tot)/3600:.2f}h | Best Dice: {best:.4f}')
print(f'✓ Checkpoint saved: {CKPT_BEST}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 6: Show final results
# ═══════════════════════════════════════════════════════════════
import torch, numpy as np
from pathlib import Path

ckpt_path = Path('/content/drive/MyDrive/ATM_Net_outputs/best_model.pth')
if not ckpt_path.exists():
    print('No checkpoint yet — run training first')
else:
    ck = torch.load(str(ckpt_path), map_location='cpu')
    print(f'='*55)
    print(f'  FINAL RESULTS')
    print(f'='*55)
    print(f'  Best epoch : {ck["epoch"]}')
    print(f'  Best Dice  : {ck["best_dice"]:.4f}')
    print()
    pc = ck.get('per_class_dice', {})
    vert = [v for k,v in pc.items() if 'Vert' in k]
    ivd  = [v for k,v in pc.items() if 'IVD'  in k]
    if vert: print(f'  Vertebrae mean Dice : {np.mean(vert):.4f}')
    if ivd:  print(f'  IVD mean Dice       : {np.mean(ivd):.4f}')
    print(f'  Overall mean Dice   : {np.mean(list(pc.values())):.4f}')
    print()
    print('  Per-class breakdown:')
    for name,d in sorted(pc.items(), key=lambda x:-x[1]):
        bar = '█' * int(d*25)
        tag = '  ★' if d>=0.90 else '  ✓' if d>=0.80 else '  ·' if d>=0.70 else ''
        print(f'    {name:<18} {d:.4f}  {bar}{tag}')